In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [ ]:
!pip install datasets soundfile librosa noisereduce webrtcvad tqdm numpy

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 66.2/66.2 kB 2.3 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done
  Created wheel for webrtcvad: filename=webrtcvad-2.0.10-cp312-cp312-linux_x86_64.whl size=73521 sha256=664aeecbdfeeaa58803f880e89fe5b4ea2f0c08b25df78f8d683bf62326f423f
  Stored in directory: /root/.cache/pip/wheels/1e/d3/95/680fa3b16848f1a58d2edaed34c496224c89a9bc63e17b3614
Successfully built webrtcvad


In [ ]:
import os
os.makedirs("/content/data/clean", exist_ok=True)

In [ ]:
from datasets import load_dataset, Audio

# Load the dataset as before
dataset = load_dataset(
    "abnerh/TORGO-database",
    split="train[:1%]"
)

# NEW: Re-cast the audio column to NOT decode by default
dataset = dataset.cast_column("audio", Audio(decode=False))

print(len(dataset))
print(dataset[0].keys())

# This will now print the keys for the *undecoded* audio data
print(dataset[0]["audio"].keys())

166
dict_keys(['audio', 'transcription', 'speech_status', 'gender', 'duration'])
dict_keys(['bytes', 'path'])


In [ ]:
import pandas as pd

# First, create the metadata from the dataset
metadata_list = []

for idx, item in enumerate(dataset):
    metadata_list.append({
        'audio_path': item['audio']['path'],
        'sentence': item.get('sentence', ''),
        'speaker_id': item.get('speaker_id', ''),
        'dysarthria_level': item.get('dysarthria_level', ''),
        'index': idx
    })

# Create DataFrame
meta_df = pd.DataFrame(metadata_list)

# Save to jsonl
metadata_path = "/content/data/clean_metadata.jsonl"
meta_df.to_json(metadata_path, orient='records', lines=True)

# Display the first few rows
display(meta_df.head())

,audio_path,sentence,speaker_id,dysarthria_level,index
0,FC01_1_arrayMic_0066.wav,,,,0
1,FC01_1_arrayMic_0016.wav,,,,1
2,FC01_1_arrayMic_0061.wav,,,,2
3,FC01_1_arrayMic_0053.wav,,,,3
4,FC01_1_arrayMic_0043.wav,,,,4


In [ ]:
import librosa
import IPython.display as ipd
import soundfile as sf
from datasets import Audio
import io

# Get the audio data directly from the dataset
audio_item = dataset[2]['audio']

print(f"Audio info: {audio_item}")

# --- CORRECTED LOGIC ---
# Check for 'bytes' FIRST
if 'bytes' in audio_item and audio_item['bytes']:
    # If the audio is stored as bytes
    print("Loading from bytes...")
    audio_bytes = audio_item['bytes']
    # Use soundfile (sf) to read from a byte stream
    audio_array, sr = sf.read(io.BytesIO(audio_bytes))

# If no bytes, THEN try the path
elif 'path' in audio_item and audio_item['path']:
    audio_path = audio_item['path']
    print(f"Loading from path: {audio_path}")
    audio_array, sr = librosa.load(audio_path, sr=None)

# If neither bytes nor path, force decode
else:
    print("Decoding audio on-the-fly...")
    dataset_decoded = dataset.cast_column("audio", Audio(decode=True))
    audio_data = dataset_decoded[2]['audio']
    audio_array = audio_data['array']
    sr = audio_data['sampling_rate']

print(f"Sample rate: {sr}, Audio length: {len(audio_array)}")

# Display the audio player
ipd.Audio(data=audio_array, rate=sr)

Audio info: {'bytes': b'RIFF$\x84\x03\x00WAVEfmt \x10\x00\x00\x00\x01\x00\x01\x00\x80>\x00\x00\x00}\x00\x00\x02\x00\x10\x00data\x00\x84\x03\x00<\x00\r\x00\xbd\xffX\x00t\xff0\x00\xfd\xff\xd4\xff\x18\x00_\xffD\x00\xb4\xff\xb6\xff\xb5\xff7\x00\xb7\xff\xb5\xffY\x00\xc6\xff%\x00\xe7\xff\xb7\xff$\x00\xd7\xff\xb4\xffk\x00\xa5\xffu\x00\xfc\xff=\x00V\x00\x97\xff\xc6\x00\x8b\xff#\x00\xf7\xff\xb9\xff\x05\x00O\x00\xd3\xff3\x00+\x00\xc4\xff\xcc\x00\x16\xffO\x00t\x00\xa2\xff9\x00\xff\xffr\x00H\x00\x9e\xff\xa9\x00\xcb\xff\xdc\xffY\x00\xd9\xffw\x00\x9a\xff0\x00\x0e\x00\xb3\xff2\x00\xf2\xff\xbf\xff/\x00\x94\xff\xe4\xff\x1d\x00G\xff\xc5\x00\x85\xff\x07\x00\xe3\xffz\xff}\x00N\xff\x8c\x00\xa4\xff\x92\x00\xbd\xff%\x001\x00\xf5\xff;\x00[\xffn\x00p\xff]\x00\x94\xff\x02\x006\x00\xe0\xff\x18\x005\x00\xe9\xff\x06\x00\x0c\x00\xa9\xff|\x00\x7f\xff2\x00\x18\x00\xd2\xff\x1d\x00\xd3\xff*\x00\xbc\xff\xfe\xff\'\x00;\x00\'\x00 \x00\xef\xff@\x00\x96\xff\xe2\xff\x08\x00\xe8\xffK\x00\xbb\xff\x8a\x00\xa1\xff\xeb\xffT\x00\x

In [ ]:
!pip install torchcodec

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.9/1.9 MB 17.4 MB/s eta 0:00:00


In [ ]:
import librosa, soundfile as sf, noisereduce, numpy as np, json, tqdm, os
import io

SAMPLE_RATE = 16000
os.makedirs("/content/data/clean", exist_ok=True)
meta = []

# Access the underlying Arrow table to get raw data
arrow_table = dataset.data.table

for i in tqdm.tqdm(range(len(dataset))):
    # Access data directly from the Arrow table
    audio_bytes = arrow_table.column("audio")[i].as_py()["bytes"]
    text = arrow_table.column("transcription")[i].as_py()

    # Read audio data from bytes
    with io.BytesIO(audio_bytes) as audio_file:
        audio_array, sr = sf.read(audio_file)

    # Preprocess
    y = librosa.resample(audio_array, orig_sr=sr, target_sr=SAMPLE_RATE)
    y, _ = librosa.effects.trim(y, top_db=30)
    noise = y[:SAMPLE_RATE//2] if len(y) > SAMPLE_RATE//2 else y
    clean = noisereduce.reduce_noise(y=y, y_noise=noise, sr=SAMPLE_RATE)
    clean = clean / (np.max(np.abs(clean)) + 1e-9)

    fname = f"sample_{i:04d}.wav"
    path = f"/content/data/clean/{fname}"
    sf.write(path, clean, SAMPLE_RATE)
    meta.append({"path": path, "text": text})

with open("/content/data/clean_metadata.jsonl", "w") as f:
    for m in meta:
        f.write(json.dumps(m) + "\n")

print(f"✅ Saved {len(meta)} clean samples.")

100%|██████████| 166/166 [00:18<00:00,  8.80it/s]

✅ Saved 166 clean samples.


In [ ]:
!apt-get update
!apt-get install ffmpeg libavcodec-dev libavutil-dev libavformat-dev libswresample-dev

Get:1 http://security.ubuntu.com/ubuntu jammy-security InRelease [129 kB]
Hit:2 http://archive.ubuntu.com/ubuntu jammy InRelease
Get:3 https://cloud.r-project.org/bin/linux/ubuntu jammy-cran40/ InRelease [3,632 B]
Hit:4 https://developer.download.nvidia.com/compute/cuda/repos/ubuntu2204/x86_64  InRelease
Hit:5 https://cli.github.com/packages stable InRelease
Get:6 http://archive.ubuntu.com/ubuntu jammy-updates InRelease [128 kB]
Get:7 https://r2u.stat.illinois.edu/ubuntu jammy InRelease [6,555 B]
Get:8 http://security.ubuntu.com/ubuntu jammy-security/main amd64 Packages [3,473 kB]
Hit:9 https://ppa.launchpadcontent.net/deadsnakes/ppa/ubuntu jammy InRelease
Hit:10 https://ppa.launchpadcontent.net/graphics-drivers/ppa/ubuntu jammy InRelease
Get:11 http://archive.ubuntu.com/ubuntu jammy-backports InRelease [127 kB]
Hit:12 https://ppa.launchpadcontent.net/ubuntugis/ppa/ubuntu jammy InRelease
Get:13 https://r2u.stat.illinois.edu/ubuntu jammy/main amd64 Packages [2,816 kB]
Get:14 https://r2u

In [ ]:
!zip -r /content/drive/MyDrive/clean_data.zip /content/data/clean /content/data/clean_metadata.jsonl

updating: content/data/clean/ (stored 0%)
updating: content/data/clean/sample_0078.wav (deflated 23%)
updating: content/data/clean/sample_0105.wav (deflated 23%)
updating: content/data/clean/sample_0030.wav (deflated 28%)
updating: content/data/clean/sample_0071.wav (deflated 12%)
updating: content/data/clean/sample_0034.wav (deflated 31%)
updating: content/data/clean/sample_0149.wav (deflated 25%)
updating: content/data/clean/sample_0126.wav (deflated 28%)
updating: content/data/clean/sample_0028.wav (deflated 25%)
updating: content/data/clean/sample_0012.wav (deflated 27%)
updating: content/data/clean/sample_0093.wav (deflated 21%)
updating: content/data/clean/sample_0062.wav (deflated 32%)
updating: content/data/clean/sample_0003.wav (deflated 26%)
updating: content/data/clean/sample_0067.wav (deflated 26%)
updating: content/data/clean/sample_0137.wav (deflated 37%)
updating: content/data/clean/sample_0033.wav (deflated 29%)
updating: content/data/clean/sample_0017.wav (deflated 13%

In [ ]:
import pandas as pd
from sklearn.model_selection import train_test_split

# Load the metadata file you just created
metadata_path = "/content/data/clean_metadata.jsonl"
meta_df = pd.read_json(metadata_path, lines=True)

# Split the data (e.g., 90% train, 10% validation)
train_df, val_df = train_test_split(meta_df, test_size=0.1, random_state=42)

print(f"Total samples: {len(meta_df)}")
print(f"Training samples: {len(train_df)}")
print(f"Validation samples: {len(val_df)}")

# --- This is the function Person B needs ---
def save_to_jsonl(dataframe, filepath):
    with open(filepath, "w") as f:
        for _, row in dataframe.iterrows():
            f.write(row.to_json() + "\n")

# --- Define the final output paths ---
# --- Define the final output paths ---
# We point this to a new folder in your "My Drive"
output_dir = "/content/drive/MyDrive/torgo_prepared"  # <-- SOLUTION
wavs_dir = f"{output_dir}/wavs"

# Create the final directories
os.makedirs(wavs_dir, exist_ok=True)

# --- Move your files to the final structure ---
# 1. Copy your clean .wav files into the 'wavs' folder
# (Your code saved them in /content/data/clean)
!cp /content/data/clean/*.wav {wavs_dir}/

# 2. Update filepaths in your DataFrames to point to the new location
# We need to remove the old '/content/data/clean/' part
train_df['path'] = train_df['path'].apply(lambda x: os.path.join(wavs_dir, os.path.basename(x)))
val_df['path'] = val_df['path'].apply(lambda x: os.path.join(wavs_dir, os.path.basename(x)))

# 3. Save the final JSONL files
save_to_jsonl(train_df, f"{output_dir}/train.jsonl")
save_to_jsonl(val_df, f"{output_dir}/val.jsonl")

print("\n✅ Successfully created 'train.jsonl' and 'val.jsonl'.")
print("✅ Copied all .wav files to 'torgo_prepared/wavs/'.")

Total samples: 166
Training samples: 149
Validation samples: 17

✅ Successfully created 'train.jsonl' and 'val.jsonl'.
✅ Copied all .wav files to 'torgo_prepared/wavs/'.


In [ ]:
 import pandas as pd
from sklearn.model_selection import train_test_split
import os
import librosa
import soundfile as sf
import noisereduce as nr
import numpy as np
import json
import tqdm
import io
from datasets import load_dataset, Audio

# --- 1. MOUNT YOUR DRIVE (If not already done) ---
from google.colab import drive
drive.mount('/content/drive')

# --- 2. DEFINE HOW MUCH DATA TO PROCESS ---
# "train[:10%]" -> 10% (Good for a first real test, maybe 5-10 mins)
# "train"       -> 100% (The full dataset. Will take ~1 hour)
DATA_SPLIT = "train"

# --- 3. DEFINE FINAL OUTPUT PATHS (in your Drive) ---
# We'll use a new folder name to be safe
output_dir = "/content/drive/MyDrive/torgo_prepared_FULL"
train_wavs_dir = os.path.join(output_dir, "train_wavs") # Separate wav folders
val_wavs_dir = os.path.join(output_dir, "val_wavs")
train_jsonl_path = os.path.join(output_dir, "train.jsonl")
val_jsonl_path = os.path.join(output_dir, "val.jsonl")

# Create all the directories
os.makedirs(train_wavs_dir, exist_ok=True)
os.makedirs(val_wavs_dir, exist_ok=True)

# --- 4. LOAD THE DATASET ---
print(f"Loading dataset split: {DATA_SPLIT}")
dataset = load_dataset("abnerh/TORGO-database", split=DATA_SPLIT)
dataset = dataset.cast_column("audio", Audio(decode=False))

# --- 5. SPLIT DATASET *BEFORE* PROCESSING ---
# This is much safer. We split the indices.
dataset_indices = list(range(len(dataset)))
train_indices, val_indices = train_test_split(dataset_indices, test_size=0.1, random_state=42)
print(f"Total samples to process: {len(dataset)}")
print(f"Training samples: {len(train_indices)}")
print(f"Validation samples: {len(val_indices)}")

# --- 6. DEFINE OUR PROCESSING FUNCTION ---
SAMPLE_RATE = 16000
def process_audio(audio_bytes):
    with io.BytesIO(audio_bytes) as audio_file:
        audio_array, orig_sr = sf.read(audio_file)

    # Resample
    y = librosa.resample(audio_array, orig_sr=orig_sr, target_sr=SAMPLE_RATE)
    # Trim silence
    y, _ = librosa.effects.trim(y, top_db=30)

    if len(y) == 0: # Handle empty audio after trimming
        return None

    # Noise reduction
    noise = y[:SAMPLE_RATE//2] if len(y) > SAMPLE_RATE//2 else y
    clean = nr.reduce_noise(y=y, y_noise=noise, sr=SAMPLE_RATE)
    # Normalize
    clean = clean / (np.max(np.abs(clean)) + 1e-9)
    return clean

# --- 7. PROCESS AND SAVE TRAINING DATA (Crash-proof) ---
print("Processing TRAINING data... (This may take a long time)")
# We will write to the .jsonl file line by line
with open(train_jsonl_path, "w") as f_train:
    for i in tqdm.tqdm(train_indices):
        item = dataset[i]

        # Process the audio
        clean_audio = process_audio(item['audio']['bytes'])
        if clean_audio is None: # Skip if audio was all silence
            continue

        text = item['transcription']
        duration = len(clean_audio) / SAMPLE_RATE

        # Create new filename and path *directly in Google Drive*
        fname = f"sample_{i:05d}.wav"
        save_path = os.path.join(train_wavs_dir, fname)

        # Save the audio file
        sf.write(save_path, clean_audio, SAMPLE_RATE)

        # Write the metadata line to our JSONL file
        metadata = {"audio_filepath": save_path, "text": text, "duration": duration}
        f_train.write(json.dumps(metadata) + "\n")

# --- 8. PROCESS AND SAVE VALIDATION DATA (Crash-proof) ---
print("Processing VALIDATION data...")
with open(val_jsonl_path, "w") as f_val:
    for i in tqdm.tqdm(val_indices):
        item = dataset[i]

        # Process the audio
        clean_audio = process_audio(item['audio']['bytes'])
        if clean_audio is None: # Skip if audio was all silence
            continue

        text = item['transcription']
        duration = len(clean_audio) / SAMPLE_RATE

        # Create new filename and path *directly in Google Drive*
        fname = f"sample_{i:05d}.wav"
        save_path = os.path.join(val_wavs_dir, fname)

        # Save the audio file
        sf.write(save_path, clean_audio, SAMPLE_RATE)

        # Write the metadata line to our JSONL file
        metadata = {"audio_filepath": save_path, "text": text, "duration": duration}
        f_val.write(json.dumps(metadata) + "\n")

print("\n🎉🎉🎉 ALL DONE! 🎉🎉🎉")
print(f"Your final, large dataset is safe in: {output_dir}")

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
Loading dataset split: train
Total samples to process: 16552
Training samples: 14896
Validation samples: 1656
Processing TRAINING data... (This may take a long time)


100%|██████████| 14896/14896 [14:49<00:00, 16.75it/s]


Processing VALIDATION data...


100%|██████████| 1656/1656 [01:37<00:00, 16.94it/s]


🎉🎉🎉 ALL DONE! 🎉🎉🎉
Your final, large dataset is safe in: /content/drive/MyDrive/torgo_prepared_FULL
